# Cancelable Interest Rate Swaps in LUSID

| Section | Topic |
|---|---|
| 1 | Instrument creation |
| 2 | Recipe |
| 3 | Portfolio and transactions |
| 4 | Valuation |
| 5 | The cancel right as an event |

## The instrument

A plain vanilla interest rate swap trades a fixed rate for a floating index on a shared notional.
Here that comes down to two legs pulling against each other:

    fixed leg      a FixedLeg paying a set rate, semi-annual
    floating leg   a FloatingLeg paying an index, no spread

On top of that, this swap carries an embedded right to cancel before maturity. LUSID models that
right as a `CancelSchedule` on the `InterestRateSwap`: give it one date and it's `European`, give
it two or more and it becomes `Bermudan`.

## The cancel schedule and valuation

A `CancelSchedule` just records that the right exists -- under `SimpleStatic`, the model this
notebook uses, it doesn't feed into how the swap gets valued. `CleanPV` and accrued interest come
out the same whether or not a `CancelSchedule` is attached, since nothing in this valuation reads
it. What attaching the schedule does give you is a
`CancelSwapEvent` you can forecast; actually exercising the right happens separately, through that
event rather than through valuation. More on this in Section 5.

---
## Setup

In [ ]:
import os
import json
import certifi
os.environ.setdefault("SSL_CERT_FILE", certifi.where())

from datetime import datetime, timezone, timedelta
import pandas as pd

import lusid
import lusid.models as m
from lusid.extensions import (
    SyncApiClientFactory, SecretsFileConfigurationLoader, EnvironmentVariablesConfigurationLoader)

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)
pd.options.display.float_format = "{:,.2f}".format

SECRETS_PATH = os.getenv("FBN_SECRETS_PATH") or (
    "secrets.json" if os.path.exists("secrets.json") else None)
config_loaders = ([SecretsFileConfigurationLoader(SECRETS_PATH)] if SECRETS_PATH
                   else [EnvironmentVariablesConfigurationLoader()])

factory = SyncApiClientFactory(config_loaders=config_loaders)


def api(cls):
    return factory.build(cls)


instruments_api   = api(lusid.InstrumentsApi)
txn_portfolio_api = api(lusid.TransactionPortfoliosApi)
portfolios_api    = api(lusid.PortfoliosApi)
quotes_api        = api(lusid.QuotesApi)
recipes_api       = api(lusid.ConfigurationRecipeApi)
aggregation_api   = api(lusid.AggregationApi)

meta = api(lusid.ApplicationMetadataApi).get_lusid_versions()
href = meta.links[0].href
print("Domain      :", href[:href.find("/app/")] if "/app/" in href else href)
print("API version :", meta.build_version)

Domain      : https://fbn-tejan.lusid.com
API version : 0.6.16597.0


---
## Configuration

Both legs use a unit `notional` of 1.0, so the swap itself is a per-unit contract -- it's the
position's own quantity that scales it up to the real traded notional. Since `SimpleStatic` just
reports the quoted mark and doesn't touch leg mechanics, no floating-rate fixings are needed here.

In [2]:
def d(year, month, day):
    return datetime(year, month, day, tzinfo=timezone.utc)


def upsert(key, name, client_internal, definition):
    """Upsert one instrument and return its LUID."""
    resp = instruments_api.upsert_instruments(scope=SCOPE, request_body={
        key: m.InstrumentDefinition(
            name=name,
            identifiers={"ClientInternal": m.InstrumentIdValue(value=client_internal)},
            definition=definition)})
    assert not resp.failed, list(resp.failed.values())[0].detail
    return resp.values[key].lusid_instrument_id


def mastered(luid):
    """Reference an instrument that already exists in the master."""
    return m.MasteredInstrument(
        instrument_type="MasteredInstrument",
        identifiers={"Instrument/default/LusidInstrumentId": luid})


def recreate_portfolio(code, display_name, base_currency, created, recipe=None):
    """Create the portfolio, replacing any earlier run so the book starts empty."""
    request = m.CreateTransactionPortfolioRequest(
        display_name=display_name, code=code, base_currency=base_currency,
        created=created, instrument_scopes=[SCOPE],
        instrument_event_configuration=None if recipe is None else
        m.InstrumentEventConfiguration(
            transaction_template_scopes=["default"],
            recipe_id=m.ResourceId(scope=SCOPE, code=recipe)))
    try:
        txn_portfolio_api.create_portfolio(
            scope=SCOPE, create_transaction_portfolio_request=request)
        print(f"Created {SCOPE}/{code}")
    except lusid.ApiException as e:
        if "PortfolioWithIdAlreadyExists" not in str(getattr(e, "body", "")):
            raise
        portfolios_api.delete_portfolio(scope=SCOPE, code=code)
        txn_portfolio_api.create_portfolio(
            scope=SCOPE, create_transaction_portfolio_request=request)
        print(f"Recreated {SCOPE}/{code}")


def upsert_price(luid, price, effective, currency):
    """One Price/mid quote, keyed on the instrument's LUID."""
    quotes_api.upsert_quotes(scope=SCOPE, request_body={
        f"{luid}-{effective:%Y%m%d}": m.UpsertQuoteRequest(
            quote_id=m.QuoteId(
                quote_series_id=m.QuoteSeriesId(
                    provider="Lusid", instrument_id=luid,
                    instrument_id_type="LusidInstrumentId",
                    quote_type="Price", field="mid"),
                effective_at=effective.isoformat()),
            metric_value=m.MetricValue(value=price, unit=currency))})


def value(portfolio, effective, metrics, currency, group_by=None):
    """Run the recipe over one portfolio and return the result as a DataFrame."""
    request = m.ValuationRequest(
        recipe_id=m.ResourceId(scope=SCOPE, code=RECIPE),
        metrics=[m.AggregateSpec(key=k, op=op) for k, op in metrics],
        group_by=group_by or ["Instrument/default/Name"],
        report_currency=currency,
        portfolio_entity_ids=[m.PortfolioEntityId(
            scope=SCOPE, code=portfolio, portfolio_entity_type="SinglePortfolio")],
        valuation_schedule=m.ValuationSchedule(effective_at=effective.isoformat()))
    return pd.DataFrame(aggregation_api.get_valuation(valuation_request=request).data)


def transactions(portfolio, from_date, as_at):
    """The portfolio's own booked transactions over a date range, as a DataFrame."""
    txns = txn_portfolio_api.get_transactions(
        scope=SCOPE, code=portfolio,
        from_transaction_date=from_date.isoformat(),
        to_transaction_date=as_at.isoformat()).values
    if not txns:
        return pd.DataFrame(columns=["date", "type", "luid", "units", "consideration"])
    return pd.DataFrame([{
        "date": pd.Timestamp(t.transaction_date).strftime("%Y-%m-%d"),
        "type": t.type,
        "luid": t.instrument_uid,
        "units": t.units,
        "consideration": t.total_consideration.amount,
    } for t in txns]).sort_values(["date", "type"]).reset_index(drop=True)


SCOPE     = "CancelableSwapDemo"
RECIPE    = "cancelable-swap-demo-recipe"
PORTFOLIO = "cancelable-swap-demo-book"

SWAP_ID   = "DEMO-CANCELSWAP-01"
DESC      = "Demo 5Y Cancelable Interest Rate Swap 4.25%"
CURRENCY  = "USD"
START     = d(2025, 3, 17)
MATURITY  = d(2030, 3, 17)
ASOF      = d(2027, 6, 15)

FIXED_RATE       = 0.0425
FIXED_FREQUENCY  = "6M"
FIXED_DAY_COUNT  = "Thirty360"

FLOAT_INDEX       = "SOFRRATE"
FLOAT_FIXING_REF  = "USD-SOFR"
FLOAT_SPREAD      = 0.0
FLOAT_FREQUENCY   = "3M"
FLOAT_DAY_COUNT   = "Actual360"

FIXED_SIDE = "Pay"
FLOAT_SIDE = "Receive"

NOTIONAL = 1.0               # unit contract -- see Configuration note above

QUANTITY = 10_000_000.00
PRICE    = 1.75              # points, quoted mark
DENOM    = 100

CALL_DATE        = d(2028, 3, 17)   # 3Y into the 5Y swap -- a single date makes this European
CALL_NOTICE_DAYS = 30

print(f"{DESC}")
print(f"  fixed   {FIXED_SIDE:<8} {FIXED_RATE:.3%}, {FIXED_FREQUENCY} {FIXED_DAY_COUNT}")
print(f"  float   {FLOAT_SIDE:<8} {FLOAT_INDEX} + {FLOAT_SPREAD:.2%} ({FLOAT_FIXING_REF})")
print(f"  callable {CALL_DATE:%Y-%m-%d}, {CALL_NOTICE_DAYS} business days' notice")
print(f"  {QUANTITY:,.0f} notional at {PRICE} points on {ASOF:%Y-%m-%d}")
print(f"  market value = {QUANTITY:,.0f} x {PRICE} / {DENOM} = {QUANTITY * PRICE / DENOM:,.2f} {CURRENCY}")

Demo 5Y Cancelable Interest Rate Swap 4.25%
  fixed   Pay      4.250%, 6M Thirty360
  float   Receive  SOFRRATE + 0.00% (USD-SOFR)
  callable 2028-03-17, 30 business days' notice
  10,000,000 notional at 1.75 points on 2027-06-15
  market value = 10,000,000 x 1.75 / 100 = 175,000.00 USD


---
# 1. Instrument creation

`legs` takes exactly two `InstrumentLeg` entries -- here one `FixedLeg` and one `FloatingLeg` -- each
with its own `notional` and `LegDefinition`. The cancel right lives separately from the legs, as a
`CancelSchedule` on the swap itself.


In [3]:
fixed_leg = m.FixedLeg(
    instrument_type="FixedLeg",
    start_date=START,
    maturity_date=MATURITY,
    notional=NOTIONAL,
    leg_definition=m.LegDefinition(
        rate_or_spread=FIXED_RATE,
        pay_receive=FIXED_SIDE,
        conventions=m.FlowConventions(
            currency=CURRENCY,
            payment_frequency=FIXED_FREQUENCY,
            day_count_convention=FIXED_DAY_COUNT,
            roll_convention=str(START.day),
            payment_calendars=[], reset_calendars=[]),
        stub_type="None",
        notional_exchange_type="None"))

floating_leg = m.FloatingLeg(
    instrument_type="FloatingLeg",
    start_date=START,
    maturity_date=MATURITY,
    notional=NOTIONAL,
    leg_definition=m.LegDefinition(
        rate_or_spread=FLOAT_SPREAD,
        pay_receive=FLOAT_SIDE,
        conventions=m.FlowConventions(
            currency=CURRENCY,
            payment_frequency=FLOAT_FREQUENCY,
            day_count_convention=FLOAT_DAY_COUNT,
            roll_convention=str(START.day),
            payment_calendars=[], reset_calendars=[]),
        index_convention=m.IndexConvention(
            currency=CURRENCY,
            payment_tenor=FLOAT_FREQUENCY,
            fixing_reference=FLOAT_FIXING_REF,
            index_name=FLOAT_INDEX,
            publication_day_lag=0,
            day_count_convention=FLOAT_DAY_COUNT),
        reset_convention="InArrears",
        stub_type="None",
        notional_exchange_type="None"))

cancel_schedule = m.CancelSchedule(
    schedule_type="CancelSchedule",
    cancel_type="European",           # a single cancel date; two or more would be "Bermudan"
    cancel_dates=[CALL_DATE],
    notice_convention=m.NoticeConvention(
        notice_days=CALL_NOTICE_DAYS,
        day_type="Business",
        calendars=[CURRENCY]))

swap = m.InterestRateSwap(
    instrument_type="InterestRateSwap",
    start_date=START,
    maturity_date=MATURITY,
    legs=[fixed_leg, floating_leg],
    cancel_schedule=cancel_schedule)

SWAP_LUID = upsert("swap", DESC, SWAP_ID, swap)
print(f"Cancelable swap : {SWAP_LUID}")

Cancelable swap : LUID_00003DFR


---
# 2. Recipe

With `SimpleStatic`, the position is priced off a quoted mark. The legs' conventions still
describe what the instrument actually is, but under this model, they don't feed into that
valuation number.

In [4]:
recipes_api.upsert_configuration_recipe(
    upsert_recipe_request=m.UpsertRecipeRequest(
        configuration_recipe=m.ConfigurationRecipe(
            scope=SCOPE, code=RECIPE,
            description="Cancelable swap, marked",
            market=m.MarketContext(
                market_rules=[m.MarketDataKeyRule(
                    key="Quote.LusidInstrumentId.*", supplier="Lusid", data_scope=SCOPE,
                    quote_type="Price", field="mid", quote_interval="1Y")],
                options=m.MarketOptions(
                    default_supplier="Lusid",
                    default_instrument_code_type="LusidInstrumentId",
                    default_scope=SCOPE)),
            pricing=m.PricingContext(
                model_rules=[m.VendorModelRule(
                    supplier="Lusid", model_name="SimpleStatic",
                    instrument_type="InterestRateSwap")],
                options=m.PricingOptions(allow_partially_successful_evaluation=True)))))

print(f"Recipe: {SCOPE}/{RECIPE}")

Recipe: CancelableSwapDemo/cancelable-swap-demo-recipe


---
# 3. Portfolio and transactions

Striking a swap doesn't involve any principal changing hands, so `totalConsideration` is zero.

In [5]:
recreate_portfolio(PORTFOLIO, "Cancelable Swap Demo Book", CURRENCY, d(2025, 1, 1), recipe=RECIPE)

txn_portfolio_api.upsert_transactions(
    scope=SCOPE, code=PORTFOLIO,
    transaction_request=[m.TransactionRequest(
        transaction_id="BUY-SWAP",
        type="Buy",
        instrument_identifiers={"Instrument/default/LusidInstrumentId": SWAP_LUID},
        transaction_date=START.isoformat(),
        settlement_date=START.isoformat(),
        units=QUANTITY,
        transaction_price=m.TransactionPrice(price=0.0, type="Price"),
        total_consideration=m.CurrencyAndAmount(amount=0.0, currency=CURRENCY),
        source="default")])

display(transactions(PORTFOLIO, START, START))

Recreated CancelableSwapDemo/cancelable-swap-demo-book


,date,type,luid,units,consideration
0,2025-03-17,Buy,LUID_00003DFR,"10,000,000.00",0.00


---
# 4. Valuation

Just one quote, at the swap's own price, expressed per unit.

In [6]:
upsert_price(SWAP_LUID, PRICE / DENOM, ASOF, CURRENCY)

METRICS = [("Instrument/default/Name", "Value"),
           ("Holding/default/Units",   "Sum"),
           ("Valuation/CleanPV",       "Sum")]

result = value(PORTFOLIO, ASOF, METRICS, CURRENCY)
display(result)

pv = result.loc[result["Instrument/default/Name"] == DESC, "Sum(Valuation/CleanPV)"].iloc[0]
print(f"LUSID CleanPV {pv:,.2f}  vs  quoted mark {QUANTITY * PRICE / DENOM:,.2f}")

,Instrument/default/Name,Sum(Holding/default/Units),Sum(Valuation/CleanPV)
0,Demo 5Y Cancelable Interest Rate Swap 4.25%,"10,000,000.00","175,000.00"


LUSID CleanPV 175,000.00  vs  quoted mark 175,000.00


---
# 5. The cancel right as an event

`instrumentEventConfiguration` has to be set on the portfolio at creation time -- passing
`recipe=RECIPE` into `recreate_portfolio()` back in section 3 is what sets it. Skip that step and
`query_applicable_instrument_events` will just quietly return zero events instead of raising an
error.

Query a window that spans the cancel date and you'll see `CancelSwapEvent` show up alongside the
swap's regular `SwapCashFlowEvent`s. The cancel right we declared back in section 1 as a static
field on the instrument now appears as something you can actually forecast.

In [7]:
events_api = api(lusid.InstrumentEventsApi)

applicable = events_api.query_applicable_instrument_events(
    query_applicable_instrument_events_request=m.QueryApplicableInstrumentEventsRequest(
        window_start=ASOF.isoformat(),
        window_end=(CALL_DATE + timedelta(days=CALL_NOTICE_DAYS + 5)).isoformat(),
        effective_at=(CALL_DATE + timedelta(days=CALL_NOTICE_DAYS + 5)).isoformat(),
        portfolio_entity_ids=[m.PortfolioEntityId(
            scope=SCOPE, code=PORTFOLIO, portfolio_entity_type="SinglePortfolio")],
        forecasting_recipe_id=m.ResourceId(scope=SCOPE, code=RECIPE))).values

display(pd.DataFrame([{
    "event type": ev.instrument_event_type,
    "eligible balance": ev.eligible_balance,
    "status": ev.instrument_event_status,
} for ev in applicable]))

,event type,eligible balance,status
0,SwapCashFlowEvent,"10,000,000.00",Active
1,SwapCashFlowEvent,"10,000,000.00",Active
2,SwapCashFlowEvent,"10,000,000.00",Active
3,SwapCashFlowEvent,"10,000,000.00",Active
4,CancelSwapEvent,"10,000,000.00",Active
5,SwapCashFlowEvent,"10,000,000.00",Active
6,SwapCashFlowEvent,"10,000,000.00",Active


---
# Summary

1. A plain vanilla interest rate swap is an `InterestRateSwap` with a `legs` list holding one
   `FixedLeg` and one `FloatingLeg`.
2. Both legs carry a unit `notional` of 1.0, which keeps the swap per-unit -- it's the position's
   own quantity that scales it up to the real traded notional.
3. It's priced off a quoted mark under `SimpleStatic`, so the legs describe the instrument rather
   than drive the number, and adding a `CancelSchedule` doesn't move `CleanPV` at all.
4. The cancel right is declared as a `CancelSchedule` on the swap, and it shows up at runtime as a
   `CancelSwapEvent` -- but only once the portfolio's own `instrumentEventConfiguration` points at
   a recipe, and that can only be set when the portfolio is created.

In [8]:
print(f"Scope      : {SCOPE}")
print(f"Portfolio  : {SCOPE}/{PORTFOLIO}")
print(f"Recipe     : {SCOPE}/{RECIPE}")
print(f"Instrument : {SWAP_LUID}")

Scope      : CancelableSwapDemo
Portfolio  : CancelableSwapDemo/cancelable-swap-demo-book
Recipe     : CancelableSwapDemo/cancelable-swap-demo-recipe
Instrument : LUID_00003DFR
